In [1]:
import pandas as pd
dataset = pd.read_csv("data/stackowerQnA_context.csv")

In [3]:
import re 
import html
from ast import literal_eval
def remove_html_tags(text):
  """
  Removes HTML tags from a string and unescapes HTML entities.

  Args:
    text: The input string containing HTML.

  Returns:
    The cleaned string without HTML tags or entities.
  """
  # 1. Compile a regular expression to find all HTML tags
  # This pattern '<[^>]+>' matches '<', followed by one or more characters
  # that are NOT '>', and then matches '>'.
  tag_re = re.compile('<[^>]+>')
  
  # 2. Use re.sub() to replace all matches of the pattern with an empty string
  no_tags = tag_re.sub('', text)
  
  # 3. Use html.unescape() to convert HTML entities (like &quot;, &lt;, &amp;)
  # back into their corresponding characters (", <, &)
  cleaned_text = html.unescape(no_tags)
  
  return cleaned_text.strip()


dataset = dataset.rename(columns={"LLM_questions": "question", "LLM_answers": "answer",
                                      "questions":"question", "answers":"answer", "contexts":"golden_context"})
dataset = dataset.dropna(subset=["question", "answer", "golden_context"])
dataset["golden_context"] = dataset["golden_context"].apply(literal_eval)
dataset = dataset.loc[dataset["golden_context"].str.len() > 0]

dataset = dataset.iloc[:min(100,len(dataset))]
dataset["question"] = dataset["question"].apply(remove_html_tags)
dataset["answer"] = dataset["answer"].apply(remove_html_tags)

In [2]:
from ast import literal_eval
dataset = dataset.rename(columns={"LLM_questions": "question", "LLM_answers": "answer"})
dataset = dataset.dropna(subset=["question", "answer", "golden_context"])
dataset["golden_context"] = dataset["golden_context"].apply(literal_eval)
dataset = dataset.loc[dataset["golden_context"].str.len() > 0]

In [7]:
dataset.iloc[1]["question"]

'I want to undersample 3 cross-validation folds from a dataset, using say, RandomUnderSampler from imblearn, and then, optimize the hyperparameters of various gbms using those undersampled folds as input.\nThe code I have so far is:\ndef train_model_with_undersampling(undersampler, estimator, scale, params, X_train, y_train):\n    # we need the resampler within a pipeline, because we are\n    # using cross-validation to optimize hyperparameters, which\n    # means that we need a left out fold without resampling to\n    # evaluate the model.\n    # The only way is with imblearn\'s pipeline (see imblearn docs)\n\n    # annoying cause we need to resample every time\n    if scale is True:\n        pipe = Pipeline([\n            ("scaler", MinMaxScaler()),\n            ("sampler", undersampler),\n            ("model", estimator),\n        ])\n    else:\n        pipe = Pipeline([\n            ("sampler", undersampler),\n            ("model", estimator),\n        ])\n\n    search = HalvingRan

In [8]:
dataset.iloc[1]["answer"]

'You can do this:\n\nGet initial folds using .split() method of your sklearn CV object. It returns indices for train and test of each fold.\n\nUndersample train fold data using imblearn sampler. You can discard resulting undersampled data, as you need only indices.\n\nExtract indices of undersampled train fold from fitted imblearn sampler and use them to get undersampled train fold indices\n\nFor each fold, save tuple (fold_train_sampled_indices, fold_test_indices)\n\n\ndef cv_undersample_split(X, y, cv, imb_sampler):\n    folds = []\n    for fold_train_idx, fold_test_idx in cv.split(X, y):\n        imb_sampler.fit_resample(X[fold_train_idx], y[fold_train_idx])\n        fold_train_sampled_idx = fold_train_idx[imb_sampler.sample_indices_]\n        folds.append((fold_train_sampled_idx, fold_test_idx))\n    return folds\n\nfolds = cv_undersample_split(X=X_train, y=y_train, \n    cv=KFold(3), \n    imb_sampler=RandomUnderSampler()\n)\n\nNow you can use folds instead of cv parameter in Halv

In [3]:
from src.rag.agentic_langgraph import AgenticLangGraph
from src.eval.evaluation import AgenticRAGEvaluator
#sklearn_hier_json = pd.read_pickle("../graph/sklearn/sklearn_with_summaries.pkl")
model_name = "gpt-oss:20b"
tool = AgenticLangGraph(model_name=model_name)

evaluator = AgenticRAGEvaluator(df=dataset, agentic_runner=tool, k_values=[3, 5, 10], context_column="golden_context")

d:\EricssonCodeGraph\hgb-rag-cqa\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
tool.run("What kind of linear models are supported by this repo?")

[INIT] Connecting to MCP servers...
[INIT] Loaded MCP tools: ['qdrant_search', 'expand_function_neighbors', 'expand_cfg_neighbors', 'functions_linked_to_issues_prs']
[INIT] Graph compiled successfully.
        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +---------+         
          | chatbot |         
          +---------+         
          .         *         
        ..           **       
       .               *      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


{'answer': '## Linear models available in this repository\n\nThe codebase bundles a comprehensive subset of **scikit‑learn’s linear‑model family**.  \nBelow is a high‑level taxonomy of the models you can import and use.\n\n| Category | Models | Typical use |\n|----------|--------|-------------|\n| **Linear regression** | `LinearRegression` (plain OLS) | Baseline continuous‑output regression |\n| | `LinearRegressionCV` | OLS with cross‑validation for `fit_intercept` & `normalize` |\n| | `ElasticNet`, `ElasticNetCV` | L1+L2 regularised regression (elastic‑net) |\n| | `Lasso`, `LassoCV` | L1 (sparse) regularised regression |\n| | `Ridge`, `RidgeCV` | L2 regularised regression |\n| | `QuantileRegressor` | Regression for conditional quantiles |\n| | `PoissonRegressor` | Count‑data (Poisson) regression |\n| | `GammaRegressor` | Positive‑skewed continuous outcomes |\n| **Logistic regression** | `LogisticRegression`, `LogisticRegressionCV` | Binary/multi‑class classification |\n| **Regularisat

In [4]:
idx = 0
row = dataset.iloc[idx]
evaluator.evaluate_single(idx, row)
evaluator.get_live_summary(idx)

[INIT] Connecting to MCP servers...
[INIT] Loaded MCP tools: ['qdrant_search', 'expand_function_neighbors', 'expand_cfg_neighbors', 'functions_linked_to_issues_prs']
[INIT] Graph compiled successfully.


{'MRR': nan, 'BLEU': nan, 'BERT': nan, 'SemSim': nan}

In [10]:
idx = 6
question = evaluator.df.iloc[idx]["question"]
answer = evaluator.df.iloc[idx]["answer"]
question

'Why did reviewers close the pull request that updated the `_check_sample_weight` error message to specify that scalar sample weights are only valid for Ridge regression?'

In [11]:
tool.run(question)

{'answer': " The PR (https://github.com/scikit-learn/scikit-learn/pull/24907) was closed because it introduced a backwards incompatible change and did not follow the guidelines for deprecating or removing features in scikit-learn.\n\nThe change made in the PR was to raise an error message when using scalar sample weights with linear regression models, other than Ridge regression, which is not allowed according to the current documentation. However, this change would break existing code that uses scalar sample weights with non-Ridge regression models and could cause unexpected behavior or errors for users who rely on these models in their workflows.\n\nIn order to prevent such issues, scikit-learn follows strict guidelines for deprecating or removing features, including providing a warning message and giving users ample time to update their code before the change is made. The PR did not follow these guidelines and was therefore closed.\n\nIt's worth noting that the pull request could st

In [23]:
for i in range(len(evaluator.df)):
    question = evaluator.df.iloc[i]["question"]
    final_state = tool.run(question)
    print(final_state)
    print("-"*20)

{'answer': " In this provided code snippet, there is no mention of a PrecisionRecallDisplay or RocCurveDisplay API. However, if we consider that these APIs are part of the same library for consistency in naming conventions:\n\n* The argument named `target_names` in PrecisionRecallDisplay might be renamed to `name` for consistency with RocCurveDisplay, which uses `class_names`.\n* But it's essential to note that this is just a guess based on common naming practices and without specific information about the API or library being used.", 'relevant_docs': [], 'relevant_node_ids': [], 'tool_log': []}
--------------------
{'answer': " Without access to the specific pull request details, it is challenging to determine the exact changes made to improve test precision for Passive-Aggressive algorithms. However, some possible improvements that could lead to better test precision might include:\n\n1. Improving the quality of training data by ensuring it is well-balanced and representative of real

KeyboardInterrupt: 

In [22]:
tool.run(question)

{'answer': ' To modify the `__sklearn_tags__()` method in both FeatureHasher and HashingVectorizer classes to correctly set the `requires_fit=False`, you would need access to the specific implementation of these classes. However, I can provide you with an example of how this change might be applied:\n\nAssuming you have the following class structure for FeatureHasher:\n```python\nclass FeatureHasher(BaseEstimator):\n    def __init__(self, n_clusters=None, random_state=None, max_features=None):\n        # Initialization code\n\n    def fit(self, X, y=None):\n        # Fit method implementation\n\n    def predict(self, X):\n        # Predict method implementation\n\n    def __sklearn_tags__(self):\n        return {"estimator": ["FeatureHasher"], "requires_fit": [False]}\n```\nTo modify the HashingVectorizer class, you could do something like this:\n```python\nclass HashingVectorizer(BaseEstimator):\n    def __init__(self, n_clusters=None, binary=True, non_negative=True, dtype=np.float32,

In [12]:
evaluator.df.iloc[idx]

statements                                                           NaN
pr_statements          ENH: Standardize parameter naming in Precision...
comments               Thanks for the PR and great that you are inter...
LLM_summaries          The pull request standardizes the `PrecisionRe...
question               What argument in the `PrecisionRecallDisplay` ...
answer                 The `estimator_name` argument was renamed to `...
LLM_scores                                                            10
golden_context         [sklearn/metrics/_plot/precision_recall_curve....
precision_3                                                         None
recall_3                                                            None
f1_3                                                                None
iou_3                                                               None
precision_5                                                         None
recall_5                                           

In [5]:
from src.utils.qdrant_store import QdrantStore
from pathlib import Path


qdrant_key_path =   "_/drant_api_key.txt"
with open(qdrant_key_path, "r") as f:
    qdrant_apikey = f.read().strip()

# Initialize your QdrantStore (read-only)
qdrant_store = QdrantStore(
    model_name="microsoft/codebert-base",
    qdrant_url="http://localhost:6333",
    collection_name="rag_collection_codebert-base_cosine",
    api_key=qdrant_apikey,
    neo4j_uri="bolt://localhost:7687",
    neo4j_auth=("neo4j", "password")
)

No sentence-transformers model found with name microsoft/codebert-base. Creating a new one with mean pooling.
d:\EricssonCodeGraph\hgb-rag-cqa\src\utils\qdrant_store.py:28: UserWarning: Api key is used with an insecure connection.
  self.client = QdrantClient(url=qdrant_url, api_key=api_key)


Successfully connected to Qdrant
Collection 'rag_collection_codebert-base_cosine' already exists.


In [6]:
query = '"_validate_data" "_check_feature_name" "_check_n_features" deprecation warnings'
top_k = 5
metadata_filter = {}
qdrant_store.search_with_scores(query, top_k=top_k, filter=metadata_filter)

[(Document(metadata={'type': 'pr_body', 'node_id': 20603, 'doc_id': '946e6931-57cc-46d5-9d43-a55176ba6664', 'chunk_size': 150, '_id': 'b6ffb940-500c-4ae3-9425-860ed44bd7a0', '_collection_name': 'rag_collection_codebert-base_cosine'}, page_content='_____________________________________ test_estimators[LinearRegression()-check_regressors_predict_single_target]'),
  0.98060906),
 (Document(metadata={'type': 'pr_body', 'node_id': 20603, 'doc_id': '24b413a9-8115-4a0a-9f8e-439905b9b649', 'chunk_size': 150, '_id': '4932a7fb-02f1-4ce3-b5c2-0a3673ac5a0c', '_collection_name': 'rag_collection_codebert-base_cosine'}, page_content='___________________________________________ test_estimators[Ridge()-check_regressors_predict_single_target]'),
  0.9805486),
 (Document(metadata={'type': 'pr_body', 'node_id': 20603, 'doc_id': 'd318b374-bcbf-4ed1-a0f0-bb5f3a0cff30', 'chunk_size': 150, '_id': '92288e04-6cdd-4fed-bb99-b5d20fcc8e08', '_collection_name': 'rag_collection_codebert-base_cosine'}, page_content='

In [26]:
from qdrant_client import models
query = 'FeatureHasher __sklearn_tags__ requires_fit False'
top_k = 10
condition = models.FieldCondition(
        key="metadata.type",
        match=models.MatchAny(any=["pr_body"]),
    )

metadata_filter = {}#models.Filter(must=[condition])
results= qdrant_store.search_with_scores(query, top_k=top_k, filter=metadata_filter)
for doc, score in results:
    print(f"Retrieved {doc.metadata['type']} document: {doc.page_content}")

Retrieved pr_title document: FIX Add ElasticNet* and LassoCV checks in feature_selection._from_model threshold
Retrieved function_name document: PolynomialFeatures.__sklearn_tags__
Retrieved pr_title document: MNT Un-xfail SplitTransformer check_estimators_pickle common test
Retrieved issue_body document: pytest -vv --pyargs sklearn -n auto -k "test_spectral_embedding_two_components"
```
Retrieved issue_body document: pytest -vv --pyargs sklearn -n auto -k "test_spectral_embedding_two_components"
```

# pip reproducer
```shell
Retrieved function_name document: CategoricalNB.__sklearn_tags__
Retrieved function_name document: MultinomialNB.__sklearn_tags__
Retrieved function_name document: test_kbinsdiscretizer_subsample_default
Retrieved function_name document: OutlierMixin.__sklearn_tags__
Retrieved function_name document: NoSampleWeightWrapper.__sklearn_tags__


In [1]:
from kg_rag import RepositoryRAG

rag = RepositoryRAG(data_dict={},qdrant_api_key="@lmafa12",qdrant_collection = "rag_collection_bart-large_cosine",model_name="facebook/bart-large")

d:\EricssonCodeGraph\hgb-rag-cqa\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No sentence-transformers model found with name facebook/bart-large. Creating a new one with mean pooling.
d:\EricssonCodeGraph\hgb-rag-cqa\utils\qdrant_store.py:26: UserWarning: Api key is used with an insecure connection.
  self.client = QdrantClient(url=qdrant_url, api_key=api_key)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Successfully connected to Qdrant
cosine
Cosine
Collection 'rag_collection_bart-large_cosine' already exists.
status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> vectors_count=None indexed_vectors_count=120188 points_count=122008 segments_count=8 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, ma

Loading checkpoint shards: 100%|██████████| 3/3 [00:05<00:00,  1.72s/it]
Some parameters are on the meta device because they were offloaded to the cpu.
Device set to use cuda:0


In [2]:
rag.search()


Retrieving top results...
Query classified as general_qa
Your query was classified as a(n) general_qa


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Retrieved Functions: _BaseDiscreteNB._update_class_log_prior, _BaseDiscreteNB.partial_fit, _asarray_with_order, _check_large_sparse, _check_partial_fit_first_call, _check_sample_weight, _determine_key_type, _ensure_sparse_format, _fit_context, _patch_raw_predict, _safe_indexing, attach_unique, capabilities, check_array, inplace_column_scale, label_binarize, load_diabetes, load_gzip_compressed_csv_data, scale, test_precomputed_kernel_not_psd, unique_labels, validate_params

Generating answer...

Answer: The provided repository appears to mainly focus on machine learning models, particularly regression and binary classification algorithms. It also seems to work with sparse matrices, as indicated by the mentions of CSR, CSC, COO, and BSR formats. However, the context does not explicitly mention specific tree-based models like Random Forest or Gradient Boosting Decision Trees. Therefore, it's not clear if tree-based models are directly included in this repository.

Retrieving top results.

In [6]:
import pandas as pd

df = pd.read_csv("./data/big_df_filtered_w_metrics_3.csv")
df.head(9)

,question,answer,issue_url,edit_functions,problem_statement,comments,pr_problem_statement,pr_comments,precision_3,recall_3,...,recall_10,f1_10,iou_10,mrr,bleu,meteor,bertscore,faithfulness,answer_relevancy,semantic_similarity
0,** What is the likely cause of the non-determi...,** The likely cause of the non-deterministic f...,https://github.com/scikit-learn/scikit-learn/i...,"['_BaseScorer._routing_repr', '_PassthroughSco...",`test_unsorted_indices` for `SVC` may fail ran...,NaN,MNT refactoring in routing _MetadataRequester\...,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.033425,0.271527,0.177282,NaN,NaN,0.451997
1,What are some possible reasons for the CI buil...,Two possible reasons could the the pip version...,https://github.com/scikit-learn/scikit-learn/i...,"['_ridge_regression', '_solve_lbfgs', 'ridge_r...",Failing CI for check_sample_weight_equivalence...,NaN,FEAT Ridge: specify individual coefficients fo...,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.007180,0.136612,-0.064510,NaN,NaN,0.329067
2,How does the order of method parameters in the...,The order of method parameters in the `_Binary...,https://github.com/scikit-learn/scikit-learn/i...,"['DecisionBoundaryDisplay.__init__', 'Decision...",MNT Make binary display method parameters' ord...,NaN,"Add title to DecisionBoundaryDisplay\nHello,\r...",NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.067487,0.226782,0.237222,NaN,NaN,0.780739
3,the question the code a technical question?,Not enough information.,https://github.com/scikit-learn/scikit-learn/i...,"['eval_and_get_f1', 'eval_and_print_metrics', ...",Add links to examples from the docstrings and ...,NaN,DOC: Add link to plot_monotonic_constraints.py...,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.004083,0.111940,-0.128630,NaN,NaN,0.199845
4,Why does the LinearRegression model struggle w...,** The LinearRegression model fails to handle ...,https://github.com/scikit-learn/scikit-learn/i...,"['LinearModel._set_intercept', '_preprocess_da...",LinearRegression on sparse matrices is not sam...,NaN,FIX the tests for convergence to the minimum n...,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.016443,0.229691,0.099623,NaN,NaN,0.613227
5,Why does chance_level_kw in RocCurveDisplay th...,It is because of how matplotlib handles ls and...,https://github.com/scikit-learn/scikit-learn/i...,"['lorenz_curve', 'lorenz_curve', 'all_displays']",`chance_level_kw` in `RocCurveDisplay` raises ...,NaN,ENH add CAP curve\n*This PR is a dupplicate fr...,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.013268,0.280783,0.181953,NaN,NaN,0.658393
6,How does the `train_test_split` function handl...,The `train_test_split` function uses `pd.qcut`...,https://github.com/scikit-learn/scikit-learn/i...,"['StratifiedShuffleSplitRegression.__init__', ...",Add balance_regression option to train_test_sp...,NaN,Stratified Split for Regression\n## Reference ...,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.052513,0.280745,0.105886,NaN,NaN,0.592891
7,How does the `KBinsDiscretizer` handle sample ...,It passes the sample weights to the `_find_bin...,https://github.com/scikit-learn/scikit-learn/i...,"['_BinMapper.fit', '_find_binning_thresholds',...",Incorrect sample weight handling in `KBinsDisc...,NaN,Added sample weight handling to BinMapper unde...,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.016982,0.325403,0.056181,NaN,NaN,0.408405
8,What is the correct condition for the kNN of a...,The correct condition is `n_neighbors <= n_sam...,https://github.com/scikit-learn/scikit-learn/i...,"['_locally_linear_embedding', 'barycenter_knei...",LocallyLinearEmbedding : n_neighbors <= n_samp...,NaN,LLE utilizing kNN for sample points\n#### Desc...,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.007330,0.162338,-0.056579,NaN,NaN,0.445815


In [23]:
q = "How does the order of method parameters in the `_BinaryClassifierCurveDisplayMixin` affect the consistency of the binary display method?"
results = rag.retriever.retrieve(query=q,top_k=10)
results

Query classified as general_qa


[]

In [22]:
from qdrant_client import models
index_filter = models.Filter(must=[
                models.FieldCondition(
                    key="metadata.type",
                    match=models.MatchValue(value="function_code")
                )
            ])
func_docs = rag.retriever.store.search(q, filter = index_filter, top_k=10)
func_docs

[Document(metadata={'type': 'function_code', 'node_id': 4951, 'doc_id': '025b7e42-f50e-4de0-9d43-dfdd6b8cef8a', '_id': '51253bd3-105e-4dc2-a93a-1237f768854b', '_collection_name': 'rag_collection'}, page_content='def test_subclassing_displays(pyplot, data, Display, params):\n    """Check that named constructors return the correct type when subclassed.\n\n    Non-regression test for:\n    https://github.com/scikit-learn/scikit-learn/pull/27675\n    """\n    X, y = data\n    estimator = DecisionTreeClassifier(random_state=0)\n\n    class SubclassOfDisplay(Display):\n        pass\n    display = SubclassOfDisplay.from_estimator(estimator, X, y, **params)\n    assert isinstance(display, SubclassOfDisplay)'),
 Document(metadata={'type': 'function_code', 'node_id': 4620, 'doc_id': '2dc69c3c-1656-435d-889e-6d265fb5f3d8', '_id': '45da84bb-dc95-41b5-b334-8e2fc7eca3b6', '_collection_name': 'rag_collection'}, page_content='def test_display_curve_error_classifier(pyplot, data, data_binary, Display):

In [25]:
rag._enrich_and_print_docs(func_docs)


Retrieved Functions: PartialDependenceDisplay.from_estimator, _average_binary_score, check_mixin_order, fit_binary, jaccard_score, test_display_curve_error_classifier, test_display_curve_error_no_response, test_display_curve_n_samples_consistency, test_subclassing_displays, test_validate_plot_params


{'functions': ['PartialDependenceDisplay.from_estimator',
  '_average_binary_score',
  'check_mixin_order',
  'fit_binary',
  'jaccard_score',
  'test_display_curve_error_classifier',
  'test_display_curve_error_no_response',
  'test_display_curve_n_samples_consistency',
  'test_subclassing_displays',
  'test_validate_plot_params'],
 'issues': [],
 'prs': []}

In [26]:
df.iloc[2].edit_functions

"['DecisionBoundaryDisplay.__init__', 'DecisionBoundaryDisplay.from_estimator', 'DecisionBoundaryDisplay.plot', 'lorenz_curve', 'lorenz_curve', 'all_displays']"

In [14]:
df.iloc[2].question

'How does the order of method parameters in the `_BinaryClassifierCurveDisplayMixin` affect the consistency of the binary display method?'